# 02 · Feature Engineering and SageMaker Feature Store

Two feature groups, keyed by `app_id`:

| Feature group | Contents | Used by |
|---|---|---|
| `steam-draft-features` | cleaned description, length, price, age rating, platforms, developer/publisher, release date, 22-tag label | Tag QA Gate, Reception |
| `steam-post-release-features` | positive/negative counts, peak CCU, playtime, owners, rating | analysis only; **never** the tag model |

The separation is the leakage guard. The training snapshot for the pipeline is queried from the **offline store**
(Athena) and written to `curated/tag_gate_snapshot/`.

In [1]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.path.expanduser("~/steam-store-copilot")]
    try:  # the kernel's working directory may have been deleted (e.g. after re-cloning)
        candidates = [os.getcwd(), os.path.dirname(os.getcwd())] + candidates
    except FileNotFoundError:
        os.chdir(os.path.expanduser("~"))
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import logging

for _name in ("sagemaker", "sagemaker.config", "botocore.credentials"):
    logging.getLogger(_name).setLevel(logging.ERROR)  # keep SDK log noise out of the outputs

import boto3
import sagemaker

from src.config import S3Layout

# Works both inside SageMaker Studio and on a laptop with Learner Lab credentials in ~/.aws/credentials
boto_session = boto3.Session(region_name=boto3.Session().region_name or "us-east-1")
sess = sagemaker.Session(boto_session=boto_session)
region = sess.boto_region_name
if os.path.exists("/opt/ml/metadata/resource-metadata.json"):  # inside SageMaker Studio
    role = sagemaker.get_execution_role(sagemaker_session=sess)
else:  # laptop with Learner Lab credentials: jobs run as the lab's execution role
    account = boto_session.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/LabRole"
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

Repo root: /Users/yatharthvardan/Documents/USD/steam-store-copilot


Region: us-east-1
Role:   arn:aws:iam::326278567375:role/LabRole
Bucket: sagemaker-us-east-1-326278567375


In [3]:
import time

import pandas as pd

from src.config import DRAFT_FEATURE_GROUP, POST_RELEASE_FEATURE_GROUP
from src.data.loaders import load_recs_games, load_store_games
from src.data.preprocess import build_tag_dataset
from src.data.s3io import fetch, write_csv
from src.features import feature_store as fs

games = load_store_games(fetch(f"{layout.raw}/store/games.csv"))
recs_games = load_recs_games(fetch(f"{layout.raw}/recs/games.csv"))
tag_ds = build_tag_dataset(games)

event_time = time.time()
draft_df = fs.draft_feature_frame(tag_ds, event_time)
post_df = fs.post_release_feature_frame(games, recs_games, tag_ds["app_id"], event_time)
print("draft:", draft_df.shape, "| post-release:", post_df.shape)
draft_df.head(3)

draft: (81927, 16) | post-release: (81927, 12)


,app_id,description,price,required_age,desc_chars,desc_words,windows,mac,linux,developers,publishers,release_date,release_year,tags_22,n_tags_22,event_time
0,496350,"Springtime, April: when the cherry trees come ...",5.24,0,1626,258,1,0,0,minori,MangaGamer,2016-07-29,2016,Adventure|Cute,2,1.792258e+09
1,1034400,"Immerse yourself in the most beloved, mystical...",4.99,0,1178,195,1,1,0,Somer Games,8floor,2019-05-06,2019,Singleplayer|Casual|2D|Puzzle|Atmospheric|Colo...,6,1.792258e+09
2,1934300,Building on the success of its acclaimed prede...,35.99,0,3637,529,1,0,0,Veitikka Studios,Matrix Games,2025-04-08,2025,Singleplayer|2D|3D|Simulation|Strategy,5,1.792258e+09


In [4]:
pd.DataFrame({"column": draft_df.columns, "dtype": draft_df.dtypes.astype(str).values})

,column,dtype
0,app_id,int64
1,description,string
2,price,float64
3,required_age,int64
4,desc_chars,int64
5,desc_words,int64
6,windows,int64
7,mac,int64
8,linux,int64
9,developers,string


## Create the feature groups
Online store enabled (low-latency lookup at draft time) plus an offline store in S3 for training snapshots.

In [5]:
draft_fg = fs.create_feature_group(
    DRAFT_FEATURE_GROUP, draft_df, layout.offline_store, role, sess,
    description="Draft-time features + 22-tag labels (no post-release signal)")
post_fg = fs.create_feature_group(
    POST_RELEASE_FEATURE_GROUP, post_df, layout.offline_store, role, sess,
    description="Post-release signals - excluded from the tag model to prevent leakage")

steam-draft-features: already exists (Created)


steam-post-release-features: already exists (Created)


In [6]:
fs.ingest(draft_fg, draft_df, max_workers=48)
fs.ingest(post_fg, post_df, max_workers=48)

  steam-draft-features: 10,000/81,927 records (35s)


  steam-draft-features: 20,000/81,927 records (58s)


  steam-draft-features: 30,000/81,927 records (81s)


  steam-draft-features: 40,000/81,927 records (105s)


  steam-draft-features: 50,000/81,927 records (128s)


...

  steam-post-release-features: 60,000/81,927 records (158s)


  steam-post-release-features: 70,000/81,927 records (180s)


  steam-post-release-features: 80,027/81,927 records (203s)


  steam-post-release-features: 81,927/81,927 records (210s)


Ingested 81,927 records into steam-post-release-features in 210s


## Online store lookup
The same game in both groups: the draft group has only what a studio knows before launch.

In [7]:
runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)
app_id = str(draft_df["app_id"].iloc[0])
for name in (DRAFT_FEATURE_GROUP, POST_RELEASE_FEATURE_GROUP):
    rec = runtime.get_record(FeatureGroupName=name, RecordIdentifierValueAsString=app_id)["Record"]
    print(f"--- {name}")
    for f in rec:
        print(f"  {f['FeatureName']:<24} {f['ValueAsString'][:80]}")

--- steam-draft-features
  app_id                   496350
  description              Springtime, April: when the cherry trees come into full bloom. The protagonist Y
  price                    5.24
  required_age             0
  desc_chars               1626
  desc_words               258
  windows                  1
  mac                      0
  linux                    0
  developers               minori
  publishers               MangaGamer
  release_date             2016-07-29
  release_year             2016
  tags_22                  Adventure|Cute
  n_tags_22                2
  event_time               1792257801.615555
--- steam-post-release-features
  app_id                   496350
  positive                 252
  negative                 3
  peak_ccu                 0
  recommendations          231
  avg_playtime_forever     8
  median_playtime_forever  8
  estimated_owners         0 - 20000
  positive_ratio           98
  user_reviews             196
  rating              

In [8]:
desc = draft_fg.describe()
print("Offline store:", desc["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"])
print("Glue table:   ", desc["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"])
pd.DataFrame(desc["FeatureDefinitions"])

Offline store: s3://sagemaker-us-east-1-326278567375/steam-store-copilot/feature-store/326278567375/sagemaker/us-east-1/offline-store/steam-draft-features-1792255901/data
Glue table:    steam_draft_features_1792255901


,FeatureName,FeatureType
0,app_id,Integral
1,description,String
2,price,Fractional
3,required_age,Integral
4,desc_chars,Integral
5,desc_words,Integral
6,windows,Integral
7,mac,Integral
8,linux,Integral
9,developers,String


## Training snapshot from the offline store (Athena)
The offline store usually becomes queryable 5-15 minutes after ingestion. The cell waits up to 20 minutes and
deduplicates to the latest record per game.

In [9]:
deadline = time.time() + 20 * 60
while not fs.offline_store_ready(draft_fg, min_rows=int(0.99 * len(draft_df))):
    if time.time() > deadline:
        raise TimeoutError("Offline store not ready yet - re-run this cell in a few minutes")
    print("waiting for offline store ...")
    time.sleep(60)

query = draft_fg.athena_query()
query.run(query_string=fs.snapshot_query(query.table_name),
          output_location=f"s3://{layout.bucket}/athena-results/")
query.wait()
snapshot = query.as_dataframe()
print(f"Snapshot rows from offline store: {len(snapshot):,}")
snapshot.head(3)

Snapshot rows from offline store: 81,927


,app_id,description,price,required_age,desc_chars,desc_words,windows,mac,linux,developers,publishers,release_date,release_year,tags_22,n_tags_22
0,1042030,Meet beautiful girls and spend time in their c...,0.49,0,465,85,1,0,0,Zloy Krot Studio,IR Studio,2019-03-22,2019,Indie|Casual|Puzzle,3
1,1388370,Heroic Armored Company is a modern tank warfar...,16.99,0,1749,273,1,0,0,Magicvalley Studio（魔法谷工作室）,Magicvalley Studio（魔法谷工作室）,2020-08-20,2020,Singleplayer|Action|Casual|3D|Simulation|First...,7
2,2751310,"Paint the map in your own color, eliminate you...",1.24,0,833,151,1,0,0,VOODOO,"QubicGames,Gamersky Games",2024-05-20,2024,Singleplayer|Action|Casual|3D|Arcade,5


In [10]:
snapshot_uri = write_csv(snapshot, f"{layout.snapshot}/snapshot.csv", index=False)
print("Training snapshot written to", snapshot_uri)
assert set(fs.DRAFT_COLUMNS) <= set(snapshot.columns)
leaky = {"positive", "negative", "peak_ccu", "avg_playtime_forever", "estimated_owners"}
assert not leaky & set(snapshot.columns), "post-release features leaked into the snapshot!"
print("Leakage check passed: no post-release columns in the tag-gate snapshot.")

Training snapshot written to s3://sagemaker-us-east-1-326278567375/steam-store-copilot/curated/tag_gate_snapshot/snapshot.csv
Leakage check passed: no post-release columns in the tag-gate snapshot.
